# Handwritten Digit Classifier

Training a CNN to classify handwritten digits (0-9). Uses the [HWD dataset](https://www.kaggle.com/datasets/metricasecuador/handwritten-digits-version-1-hwd-v1) via kagglehub when available, with MNIST as fallback.

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import torchvision
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from tqdm.notebook import tqdm
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Download Dataset

Attempts to download the HWD dataset via kagglehub. Falls back to MNIST if unavailable.

In [ ]:
USE_KAGGLE = False
try:
    import kagglehub
    path = kagglehub.dataset_download('metricasecuador/handwritten-digits-version-1-hwd-v1')
    print(f'HWD dataset downloaded to: {path}')
    USE_KAGGLE = True

    # Find CSV files
    all_files = []
    for root, dirs, files in os.walk(path):
        for f in files:
            all_files.append(os.path.join(root, f))
    print('Files found:')
    for f in all_files[:20]:
        print(f'  {f}')
except Exception as e:
    print(f'Could not download HWD dataset: {e}')
    print('Falling back to MNIST...')

if not USE_KAGGLE:
    train_data = torchvision.datasets.MNIST(root='./data', train=True, download=True)
    test_data = torchvision.datasets.MNIST(root='./data', train=False, download=True)
    print(f'MNIST loaded: {len(train_data)} train, {len(test_data)} test samples')

## 2. Data Exploration

In [ ]:
if USE_KAGGLE:
    csv_path = [f for f in all_files if f.endswith('.csv')]
    if csv_path:
        df = pd.read_csv(csv_path[0])
    print(f'Dataset shape: {df.shape}')
    print(f'Columns: {df.columns.tolist()[:10]}... ({len(df.columns)} total)')
    print(f'Null values: {df.isnull().sum().sum()}')

    # Separate labels and pixel data
    label_col = [c for c in df.columns if 'label' in c.lower() or 'class' in c.lower() or 'digit' in c.lower() or 'target' in c.lower()]
    if label_col:
        labels = df[label_col[0]].values
        pixels = df.drop(columns=label_col).values
    else:
        labels = df.iloc[:, 0].values
        pixels = df.iloc[:, 1:].values

    n_pixels = pixels.shape[1]
    img_size = int(np.sqrt(n_pixels))
    print(f'Image size: {img_size}x{img_size}, Pixel range: [{pixels.min()}, {pixels.max()}]')
else:
    labels = train_data.targets.numpy()
    pixels = train_data.data.numpy().reshape(-1, 784)
    img_size = 28
    test_labels = test_data.targets.numpy()
    test_pixels = test_data.data.numpy().reshape(-1, 784)
    print(f'Train: {pixels.shape}, Test: {test_pixels.shape}')
    print(f'Image size: {img_size}x{img_size}, Pixel range: [{pixels.min()}, {pixels.max()}]')

print(f'Unique labels: {np.unique(labels)}')

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(10, 5))
unique, counts = np.unique(labels, return_counts=True)
ax.bar(unique, counts, color='#f97316', edgecolor='#ea580c')
ax.set_xlabel('Digit', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Class Distribution', fontsize=14)
ax.set_xticks(range(10))
for u, c in zip(unique, counts):
    ax.text(u, c + max(counts)*0.01, str(c), ha='center', fontsize=9)
plt.tight_layout()
plt.show()
print(f'Total samples: {len(labels)}')

In [ ]:
# Sample images per digit
fig, axes = plt.subplots(2, 10, figsize=(15, 4))
for digit in range(10):
    digit_indices = np.where(labels == digit)[0]
    for row in range(2):
        idx = digit_indices[row]
        img = pixels[idx].reshape(img_size, img_size)
        axes[row, digit].imshow(img, cmap='gray')
        axes[row, digit].axis('off')
        if row == 0:
            axes[row, digit].set_title(str(digit), fontsize=12, fontweight='bold')
plt.suptitle('Sample Images per Digit', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Pixel intensity distribution and mean images
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pixels.flatten(), bins=50, color='#f97316', alpha=0.8)
ax.set_title('Pixel Intensity Distribution')
ax.set_xlabel('Pixel Value')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for digit in range(10):
    mean_img = pixels[labels == digit].mean(axis=0).reshape(img_size, img_size)
    axes[digit].imshow(mean_img, cmap='hot')
    axes[digit].axis('off')
    axes[digit].set_title(str(digit))
plt.suptitle('Mean Image per Digit', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Normalize to [0, 1] and resize to 28x28 if needed
pixel_max = pixels.max()
pixels_norm = pixels.astype(np.float32) / pixel_max

if img_size != 28:
    from PIL import Image
    resized = []
    for i in range(len(pixels_norm)):
        img = Image.fromarray((pixels_norm[i].reshape(img_size, img_size) * 255).astype(np.uint8))
        img = img.resize((28, 28), Image.LANCZOS)
        resized.append(np.array(img, dtype=np.float32) / 255.0)
    pixels_norm = np.array(resized)

X = torch.tensor(pixels_norm.reshape(-1, 1, 28, 28))
y = torch.tensor(labels.astype(np.int64))

if not USE_KAGGLE:
    # Also prepare test data
    test_norm = test_pixels.astype(np.float32) / pixel_max
    X_test = torch.tensor(test_norm.reshape(-1, 1, 28, 28))
    y_test = torch.tensor(test_labels.astype(np.int64))

print(f'X shape: {X.shape}, range: [{X.min():.3f}, {X.max():.3f}]')

In [ ]:
# Train/Val/Test split
torch.manual_seed(42)

if USE_KAGGLE:
    dataset = TensorDataset(X, y)
    n = len(dataset)
    n_train = int(0.8 * n)
    n_val = int(0.1 * n)
    n_test = n - n_train - n_val
    train_set, val_set, test_set = random_split(dataset, [n_train, n_val, n_test])
else:
    full_train = TensorDataset(X, y)
    n = len(full_train)
    n_train = int(0.9 * n)
    n_val = n - n_train
    train_set, val_set = random_split(full_train, [n_train, n_val])
    test_set = TensorDataset(X_test, y_test)

print(f'Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}')

BATCH_SIZE = 128
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## 4. Model Architecture

CNN with batch normalization and dropout for robust digit classification.

In [ ]:
class DigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 1 -> 32 channels
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            # Block 2: 32 -> 64 channels
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            # Block 3: 64 -> 128 channels
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = DigitCNN().to(device)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

## 5. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

NUM_EPOCHS = 30
best_val_acc = 0
patience_limit = 7
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for X_batch, y_batch in pbar:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)
        _, predicted = outputs.max(1)
        train_total += y_batch.size(0)
        train_correct += predicted.eq(y_batch).sum().item()
        pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{100.*train_correct/train_total:.1f}%')

    train_loss /= train_total
    train_acc = train_correct / train_total

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)
            _, predicted = outputs.max(1)
            val_total += y_batch.size(0)
            val_correct += predicted.eq(y_batch).sum().item()

    val_loss /= val_total
    val_acc = val_correct / val_total
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch+1}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, '
          f'val_loss={val_loss:.4f}, val_acc={val_acc:.4f}, lr={optimizer.param_groups[0]["lr"]:.6f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'  -> New best model saved! (val_acc={val_acc:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= patience_limit:
            print(f'Early stopping at epoch {epoch+1}')
            break

print(f'\nBest validation accuracy: {best_val_acc:.4f}')

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train', color='#f97316', linewidth=2)
ax1.plot(history['val_loss'], label='Validation', color='#3b82f6', linewidth=2)
ax1.set_title('Loss', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], label='Train', color='#f97316', linewidth=2)
ax2.plot(history['val_acc'], label='Validation', color='#3b82f6', linewidth=2)
ax2.set_title('Accuracy', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Evaluation

In [ ]:
model.load_state_dict(torch.load('best_model.pth', weights_only=True))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
test_acc = (all_preds == all_labels).mean()
print(f'Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')

In [ ]:
print(classification_report(all_labels, all_preds, target_names=[str(i) for i in range(10)]))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=range(10), yticklabels=range(10))
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
sample_loader = DataLoader(test_set, batch_size=20, shuffle=True)
X_samples, y_samples = next(iter(sample_loader))

model.eval()
with torch.no_grad():
    preds = model(X_samples.to(device)).argmax(1).cpu()

for i in range(20):
    row, col = divmod(i, 10)
    axes[row, col].imshow(X_samples[i].squeeze().numpy(), cmap='gray')
    axes[row, col].axis('off')
    color = 'green' if preds[i] == y_samples[i] else 'red'
    axes[row, col].set_title(f'{preds[i].item()}', color=color, fontsize=11, fontweight='bold')

plt.suptitle('Predictions (green=correct, red=wrong)', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Export to ONNX

Export the trained model to ONNX format for browser inference with ONNX Runtime Web.

In [ ]:
import onnx
import onnxruntime as ort

model.eval()
model_cpu = model.cpu()
dummy_input = torch.randn(1, 1, 28, 28)

onnx_path = 'digit_classifier.onnx'
torch.onnx.export(
    model_cpu, dummy_input, onnx_path,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    opset_version=13
)

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print(f'ONNX model exported: {os.path.getsize(onnx_path) / 1024:.1f} KB')

# Validate outputs match
ort_session = ort.InferenceSession(onnx_path)
sample = X_samples[0:1].numpy()
with torch.no_grad():
    pt_out = model_cpu(torch.tensor(sample)).numpy()
onnx_out = ort_session.run(None, {'input': sample})[0]
print(f'Max output difference: {np.abs(pt_out - onnx_out).max():.8f}')
print('Validation passed!' if np.allclose(pt_out, onnx_out, atol=1e-5) else 'WARNING: Outputs differ!')

In [ ]:
# Copy model to web static directory
import shutil

web_model_dir = os.path.join('..', 'web', 'static', 'model')
os.makedirs(web_model_dir, exist_ok=True)
shutil.copy2(onnx_path, os.path.join(web_model_dir, 'digit_classifier.onnx'))
print(f'Model copied to {web_model_dir}/digit_classifier.onnx')
print('\nDone! The model is ready for browser inference.')